# 01 — The TCI Feature Contract

**What this notebook does.** Establishes what the Threat Confidence Index actually
consumes, verifies the deployed booster against that contract, and reproduces the
train/serve skew that made the previous system alarm on an empty room.

Nothing here trains a model. It is an audit of what already exists.

## Why a contract is needed

Four mutually inconsistent definitions of "the fusion inputs" existed in this
repository at once:

| Where | Features |
|---|---|
| Deployed booster + serving call | 5 — vision, audio, motion, identity, is_night |
| `FusionEngine.WEIGHTS` (fallback only) | 7 — + behaviour, weapon, fire |
| Retraining notebook's intended schema | 7 — violence, anomaly, audio, weapon, fire, motion, identity |
| Mid-semester report section 3.2 | 8 — vision, audio, motion, behaviour, identity, weapon, fire, is_night |

A model trained against any one of them produces something the serving path cannot
consume. `sentrix/fusion/features.py` now makes the contract explicit and versioned.

In [ ]:
import sys, warnings, json
from pathlib import Path
warnings.filterwarnings("ignore")

ROOT = Path.cwd()
while not (ROOT / "sentrix").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
print("repo root:", ROOT)

import numpy as np
import pandas as pd

In [ ]:
from sentrix.fusion.features import CONTRACT_V1, CONTRACT_V2, FEATURE_DOCS, FeatureVector

print(f"CONTRACT_V2 (canonical, {len(CONTRACT_V2)} features)")
for i, name in enumerate(CONTRACT_V2):
    print(f"  {i}. {name:<10} {FEATURE_DOCS[name]}")

print(f"\nCONTRACT_V1 (legacy shipped booster, {len(CONTRACT_V1)} features)")
print("  ", ", ".join(CONTRACT_V1))
print("\nMissing from V1:", sorted(set(CONTRACT_V2) - set(CONTRACT_V1)))

## Which contract does the shipped model use?

`TCIEngine` reads `booster.num_features()` and selects the contract automatically,
so promoting a retrained 8-feature model needs no code change.

In [ ]:
import xgboost as xgb
from sentrix.paths import MODELS_DIR

booster = xgb.Booster()
booster.load_model(str(MODELS_DIR / "tci_xgboost.json"))
n = booster.num_features()
print("booster expects", n, "features -> contract",
      "v1" if n == 5 else "v2" if n == 8 else "UNKNOWN")

## Reproducing the train/serve skew

`training/scripts/refit_xgboost.py` fits on **raw** scores spanning `[0, 1]`.
The previous serving path pushed every score through

```python
def calibrate_score(self, raw_score, A=-1.5, B=0.5):
    """Platt scaling (dummy params for capstone demonstration)"""
    return 1.0 / (1.0 + np.exp(A * raw_score + B))
```

Those constants were never fitted — the docstring says so. The function maps
`[0, 1]` onto roughly `[0.378, 0.731]`, so the model never saw an input resembling
its own training distribution.

In [ ]:
def old_calibrate(x, A=-1.5, B=0.5):
    return 1.0 / (1.0 + np.exp(A * x + B))

grid = np.linspace(0, 1, 11)
print("raw -> 'calibrated' mapping")
for x in grid[::2]:
    print(f"  {x:.2f} -> {old_calibrate(x):.3f}")
print(f"\nfull output range: [{old_calibrate(0):.3f}, {old_calibrate(1):.3f}]")
print("training input range: [0.000, 1.000]")

In [ ]:
scenarios = {
    "empty room, day":    [0.0, 0.0, 0.0, 0.0, 0.0],
    "empty room, night":  [0.0, 0.0, 0.0, 0.0, 1.0],
    "mid activity":       [0.5, 0.5, 0.5, 0.5, 0.0],
    "high threat":        [0.9, 0.9, 0.9, 0.1, 1.0],
}

rows = []
for name, v in scenarios.items():
    raw = np.array([v], dtype=np.float32)
    cal = np.array([[old_calibrate(v[0]), old_calibrate(v[1]),
                     old_calibrate(v[2]), old_calibrate(v[3]), v[4]]], dtype=np.float32)
    rows.append({
        "scenario": name,
        "as trained (raw)":     round(float(booster.predict(xgb.DMatrix(raw))[0]), 4),
        "as served (old code)": round(float(booster.predict(xgb.DMatrix(cal))[0]), 4),
    })
df = pd.DataFrame(rows)
df["inflation"] = (df["as served (old code)"] - df["as trained (raw)"]).round(4)
df

An empty room scored **0.425** instead of 0.012 — level 2 `SUSPICIOUS`, permanently.
Because the previous escalation ladder sent an SMS at level 2 and sounded the siren
at level 3, that single defect is the direct cause of the 1,489 evidence bundles and
2,415 alert snapshots found in `backend/runtime/`.

Note also the ceiling: a genuine high-threat scene was *pulled down* from 1.00 to
0.84. The usable range of TCI had collapsed to about `[0.40, 0.84]`.

## The fixed path

`FusionConfig.calibrate_inputs` now defaults to `False`, so raw scores reach the
model exactly as it was fitted.

In [ ]:
from sentrix.fusion.tci import TCIEngine
from sentrix.fusion.policy import LevelPolicy

def steady(fv, seconds=12.0, hz=6.0):
    engine, policy, t = TCIEngine(), LevelPolicy(), 0.0
    for _ in range(int(seconds * hz)):
        t += 1.0 / hz
        r = engine.compute(fv, now=t)
        st = policy.update(r.tci, now=t)
    return r, st

cases = {
    "empty room (day)":       FeatureVector(),
    "empty room (night)":     FeatureVector(is_night=1.0),
    "unknown person, still":  FeatureVector(vision=.85, motion=.15, identity=.6, person_present=True),
    "unknown person walking": FeatureVector(vision=.85, motion=.35, identity=.6, behaviour=.2,
                                            person_present=True, behaviour_label="walking"),
    "enrolled resident":      FeatureVector(vision=.88, motion=.2, person_present=True, authorized=True),
    "fight in progress":      FeatureVector(vision=.9, motion=.6, identity=.6, behaviour=.88,
                                            audio=.72, person_present=True, behaviour_label="fighting"),
    "knife visible":          FeatureVector(vision=.88, motion=.3, identity=.6, weapon=.81, person_present=True),
    "fire detected":          FeatureVector(motion=.4, fire=.88),
}
out = []
for name, fv in cases.items():
    r, st = steady(fv)
    out.append({"scenario": name, "TCI": round(r.tci, 3),
                "level": st.level, "status": st.status, "source": r.source})
pd.DataFrame(out)

## Sensitivity: which feature actually moves TCI?

A one-at-a-time sweep. A flat line means a modality the score cannot see — which is
exactly what weapon, fire and behaviour looked like before, because the 5-feature
booster has no input for them.

In [ ]:
import matplotlib.pyplot as plt
from sentrix.config import settings

sweep = np.linspace(0, 1, 21)
fig, ax = plt.subplots(figsize=(9, 5))
for feat in CONTRACT_V2:
    if feat == "is_night":
        continue
    ys = []
    for value in sweep:
        engine = TCIEngine()      # fresh engine: no EMA carry-over between points
        fv = FeatureVector(**{feat: float(value)}, person_present=True)
        ys.append(engine.compute(fv, now=1.0).raw_tci)
    ax.plot(sweep, ys, marker="o", ms=3, label=feat)

for t in settings.policy.thresholds:
    ax.axhline(t, color="grey", lw=0.6, ls="--")
ax.set_xlabel("feature value"); ax.set_ylabel("raw TCI")
ax.set_title("One-at-a-time sensitivity (dashed = level bands)")
ax.legend(ncol=4, fontsize=8); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

## Where to go next

The 5-feature booster is still synthetic — `refit_xgboost.py` generates its own
labels from a formula, so its reported `R-squared = 0.984` measures nothing except
that XGBoost can memorise that formula.

Running the pipeline over labelled clips writes real feature vectors to
`var/events/*.jsonl`. **02_tci_calibration.ipynb** consumes those and turns TCI
into a calibrated probability; **03_tci_thresholds.ipynb** derives the level bands
from a cost matrix rather than asserting them.